In [1]:
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [2]:
llm = ChatGroq(
        model="qwen/qwen3.8-27b", 
        
)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [3]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('research_paper_1.pdf')
docs = loader.load()
docs[:2]

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'source': 'research_paper_1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Published as a conference paper at ICLR 2025\nAGENTIC AI FOR SCIENTIFIC DISCOVERY : A S URVEY\nOF PROGRESS , C HALLENGES , AND FUTURE DIREC -\nTIONS\nMourad Gridach, Jay Nanavati, Khaldoun Zine El Abidine, Lenon Mendes & Christina Mack\nIQVIA\n{firstname.lastname}@iqvia.com\nABSTRACT\nThe integration of Agentic AI into scientific discovery marks a new frontier in\nresearch automation. These AI systems, capable of reasoning, planning, and au-\ntonomous decision-making, are transforming how scientists perform literature re-\nview, generate hypotheses, conduct experiments, and analyze results. This sur-\nvey provides a comprehensive overview of Agentic AI for scientific discovery,\ncategorizing existing systems and tools, and highlighting recent progress across\nfields such as chemistry, biology,

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
splits = splitter.split_documents(docs)
splits[:5]

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'source': 'research_paper_1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Published as a conference paper at ICLR 2025\nAGENTIC AI FOR SCIENTIFIC DISCOVERY : A S URVEY\nOF PROGRESS , C HALLENGES , AND FUTURE DIREC -\nTIONS\nMourad Gridach, Jay Nanavati, Khaldoun Zine El Abidine, Lenon Mendes & Christina Mack\nIQVIA\n{firstname.lastname}@iqvia.com\nABSTRACT\nThe integration of Agentic AI into scientific discovery marks a new frontier in\nresearch automation. These AI systems, capable of reasoning, planning, and au-\ntonomous decision-making, are transforming how scientists perform literature re-\nview, generate hypotheses, conduct experiments, and analyze results. This sur-\nvey provides a comprehensive overview of Agentic AI for scientific discovery,\ncategorizing existing systems and tools, and highlighting recent progress across\nfields such as chemistry, biology,

In [5]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model
)

vector_retriever = vectorstore.as_retriever(kwargs={
    'k':2
})

In [7]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 2

In [8]:
from langchain_classic.retrievers import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever,vector_retriever],
    weights=[0.5,0.5]
)

In [9]:
# Helper: run ONE query through all three retrievers and show what each returns
def compare_retrievers(query):
    print("QUERY:", query)
    print("=" * 72)
    for name, r in [("BM25   (sparse / keyword)", bm25_retriever),
                    ("Vector (dense  / semantic)", vector_retriever),
                    ("Hybrid (RRF fusion)       ", hybrid_retriever)]:
        docs = r.invoke(query)
        pages = [d.metadata.get("page", "?") for d in docs]
        print(f"\n[{name}]  pages={pages}")
        for d in docs:
            snippet = " ".join(d.page_content.split())[:1000]
            print("   -", snippet, "...")
    print()

In [10]:
# Pick a rare exact term that ACTUALLY occurs in this PDF (so the demo always works).
# Proper-noun-like, low-frequency tokens are exactly what dense retrieval tends to miss.
import re
from collections import Counter
tokens = re.findall(r"[A-Za-z][A-Za-z'\-]{4,}", " ".join(s.page_content for s in splits))
freq = Counter(t.lower() for t in tokens)
candidates = sorted({t for t in tokens if t[0].isupper() and freq[t.lower()] <= 2})
print("Candidate rare exact terms found in the corpus:")
print(candidates[:20])


Candidate rare exact terms found in the corpus:
['ABSTRACT', 'AI-driven', 'AI-human', 'AI-made', 'AI-powered', 'AXONOMY', 'Abbasian', 'Abidine', 'Abramovich', 'Achiam', 'Additionally', 'AgentIn', 'Ajith', 'Another', 'Aristotle', 'BioPlanner', 'CALMS', 'CHARACTERISTICS', 'CONCEPTS', 'CRISPR-GPT']


In [12]:
# Query A - EXACT rare term. Watch BM25 surface the chunk containing it,
# while the dense retriever may rank it lower or miss it entirely.
keyword_query = candidates[0] if candidates else "the exact term"
compare_retrievers(keyword_query)


QUERY: ABSTRACT

[BM25   (sparse / keyword)]  pages=[0, 5]
   - Published as a conference paper at ICLR 2025 AGENTIC AI FOR SCIENTIFIC DISCOVERY : A S URVEY OF PROGRESS , C HALLENGES , AND FUTURE DIREC - TIONS Mourad Gridach, Jay Nanavati, Khaldoun Zine El Abidine, Lenon Mendes & Christina Mack IQVIA {firstname.lastname}@iqvia.com ABSTRACT The integration of Agentic AI into scientific discovery marks a new frontier in research automation. These AI systems, capable of reasoning, planning, and au- tonomous decision-making, are transforming how scientists perform literature re- view, generate hypotheses, conduct experiments, and analyze results. This sur- vey provides a comprehensive overview of Agentic AI for scientific discovery, categorizing existing systems and tools, and highlighting recent progress across fields such as chemistry, biology, and materials science. We discuss key evalua- tion metrics, implementation frameworks, and commonly used datasets to offer a detailed understandi

In [13]:
# Query B - PURE PARAPHRASE (no keyword overlap with the document wording).
# Here BM25 struggles and the dense retriever wins on meaning.
compare_retrievers("how can software systems make decisions and act on their own?")


QUERY: how can software systems make decisions and act on their own?

[BM25   (sparse / keyword)]  pages=[1, 3]
   - ery Schmidgall et al. (2025); Baek et al. (2024); Ghafarollahi & Buehler (2024b); Swanson et al. (2024); Xiao et al. (2024), software development Qian et al. (2024); White (2024) and healthcare Tang et al. (2024); Kim et al. (2024). While multi-agent systems are powerful in solving difficult problems in complex environments, the communication and interaction between agents remain one of the challenges compared to single agent systems. 3 T AXONOMY OF AGENTIC AI FOR SCIENTIFIC DISCOVERY The scope of Agentic AI for scientific discovery is vast, encompassing tasks such as hypothesis gen- eration, experiment design, data analysis, and literature review. By automating these traditionally labor-intensive processes, Agentic AI has the potential to accelerate the pace of scientific discov- ery, reduce costs, and democratize access to advanced research tools. However, the true pow

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template( """
        Respond to the user query based on the given context.
        If the context is not relevant to the query say context is not suffiecient to 
        answer this query and do not answer the query. 
        Query: {query}, context:{context}"""
    )
chain = prompt | llm | StrOutputParser()


In [18]:
while True:
   query=input("query:")
   if query=="exit":
        break
   print("retrieved context:",vector_retriever.invoke(query))
   print(chain.invoke({"query":query,
             "context":vector_retriever.invoke(query)}))


retrieved context: [Document(metadata={'page_label': '2', 'creationdate': 'D:20260117133359', 'page': 1, 'total_pages': 6, 'producer': 'PDFium', 'creator': 'PDFium', 'source': 'research_paper_1.pdf'}, page_content='Published as a conference paper at ICLR 2025\n2 A GENTIC AI: F OUNDATIONS AND KEY CONCEPTS\n2.1 D EFINITION AND CHARACTERISTICS\nThe concept of an “agent” has a rich history and has been explored across various disciplines in-\ncluding philosophy. It has been discussed by influential philosophers starting from Aristotle to\nHume among others. Generally, an “agent” is an entity that has the ability to act, while the con-\ncept of “agency” refers to the exercise or representation of this ability Schlosser (2019). In artificial\nintelligence, an agent is an autonomous intelligent entity capable of performing appropriate and\ncontextually relevant actions in response to sensory input, whether operating in physical, virtual, or\nmixed-reality environments. Agentic AI introduces a

In [22]:
while True:
   query=input("query:")
   if query=="exit":
        break
   print("retrieved context:",bm25_retriever.invoke(query))
   print(chain.invoke({"query":query,
             "context":bm25_retriever.invoke(query)}))


retrieved context: [Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'source': 'research_paper_1.pdf', 'total_pages': 6, 'page': 2, 'page_label': '3'}, page_content='Published as a conference paper at ICLR 2025\nproblem-solving. As the field continues to evolve, its applications in scientific discovery are ex-\npanding across diverse domains, from chemistry and biology to materials science and healthcare.\nAgentic AI systems can be broadly categorized based on their level of autonomy, interaction with\nresearchers, and scope of application.\n3.1 F ULLY AUTONOMOUS SYSTEMS\nFully autonomous systems are designed to operate independently, automating end-to-end scientific\nworkflows with minimal human intervention. These systems leverage advanced AI capabilities,\nsuch as natural language understanding, planning, and decision-making, to perform complex tasks\nranging from hypothesis generation to experiment execution. Boiko et al. (2023) deve

In [23]:
while True:
   query=input("query:")
   if query=="exit":
        break
   print("retrieved context:",hybrid_retriever.invoke(query))
   print(chain.invoke({"query":query,
             "context":hybrid_retriever.invoke(query)}))


retrieved context: [Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'source': 'research_paper_1.pdf', 'total_pages': 6, 'page': 5, 'page_label': '6'}, page_content='frameworks for scientific discovery in various domains.\nFigure 1: Agentic AI workflow for scien-\ntific discovery.\nFigure 2: AI Agents frameworks for scientific dis-\ncovery.\n6 I MPLEMENTATION TOOLS , DATASETS AND METRICS\nThe development and evaluation of agentic AI systems for scientific discovery rely on a robust tools,\ncurated datasets, and well-defined evaluation metrics. This section provides an overview of the key\nresources used in the field to facilitate the design, training, and assessment of autonomous AI agents\nfor scientific discovery.\n6'), Document(metadata={'total_pages': 6, 'page_label': '5', 'source': 'research_paper_1.pdf', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'page': 4, 'producer': 'PDFium'}, page_content='Published as a conferenc